# 🌿 Chemigran TCM — YOLOv8 Tongue Diagnosis Training

**Shezhen (舌诊) Dataset** — 21 tongue feature classes, ~3,000 images

### Steps:
1. Runtime → Change runtime type → **T4 GPU** (free)
2. Run all cells in order
3. Download `tongue_yolo_best.pt` when done
4. Place it in `tcm_platform/models/tongue_yolo_best.pt`

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Cell 2: Install ultralytics ───────────────────────────────────────────────
!pip install ultralytics -q
from ultralytics import YOLO
print("✓ ultralytics ready")

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# List top-level Drive contents to find your folder
print("Drive contents:")
for f in os.listdir('/content/drive/MyDrive'):
    print(' ', f)

In [ ]:
# ── Cell 4: Locate the shezhen_data folder ────────────────────────────────────
# UPDATE THIS PATH if your folder is named differently or in a subfolder
# e.g. '/content/drive/MyDrive/tcm_platform/shezhen_data'
#      '/content/drive/MyDrive/shezhen_data'

import os
from pathlib import Path

# Auto-search for shezhen_data in Drive
DRIVE_ROOT = Path('/content/drive/MyDrive')
shezhen_data = None

for p in DRIVE_ROOT.rglob('shezhen_data'):
    if p.is_dir():
        shezhen_data = p
        print(f'Found: {p}')
        break

if shezhen_data is None:
    # Manual fallback — set this to where you uploaded shezhen_data
    shezhen_data = DRIVE_ROOT / 'shezhen_data'
    print(f'Not auto-found — using: {shezhen_data}')
    print('Update DRIVE_ROOT path above if needed')

print(f'shezhen_data path: {shezhen_data}')
print('Contents:', list(shezhen_data.iterdir()) if shezhen_data.exists() else 'NOT FOUND')

In [ ]:
# ── Cell 5: Copy dataset to /content/ (much faster than training from Drive) ──
import shutil

LOCAL = Path('/content/shezhen_data')
if LOCAL.exists():
    shutil.rmtree(LOCAL)

print('Copying dataset from Drive to /content/ (faster I/O during training)...')
shutil.copytree(shezhen_data, LOCAL)
print(f'Done. Size: {sum(f.stat().st_size for f in LOCAL.rglob("*") if f.is_file()) / 1e6:.0f} MB')

In [ ]:
# ── Cell 6: Build dataset.yaml with correct Colab paths ──────────────────────
import json
import random
from pathlib import Path
from collections import defaultdict

LOCAL = Path('/content/shezhen_data')

CLASS_NAMES = [
    'jiankangshe','botaishe','hongshe','zishe','pangdashe','shoushe',
    'hongdianshe','liewenshe','chihenshe','baitaishe','huangtaishe','heitaishe',
    'huataishe','shenquao','shenqutu','gandanao','gandantu','piweiao',
    'piweitu','xinfeiao','xinfeitu'
]

# Find dataset root (handles any nesting in the zip)
coco_root = None
for p in LOCAL.rglob('train'):
    if (p / 'annotations' / 'train.json').exists():
        coco_root = p.parent
        break

if coco_root is None:
    # Maybe already flat
    for p in LOCAL.rglob('train.json'):
        coco_root = p.parent.parent.parent
        break

print(f'Dataset root: {coco_root}')

train_img_dir = coco_root / 'train' / 'images'
train_lbl_dir = coco_root / 'train' / 'labels'
test_img_dir  = coco_root / 'test'  / 'images'
test_lbl_dir  = coco_root / 'test'  / 'labels'

img_exts = {'.jpg', '.jpeg', '.png'}

# Convert COCO → YOLO for train if labels not yet present
if not train_lbl_dir.exists() or len(list(train_lbl_dir.glob('*.txt'))) < 100:
    print('Converting COCO → YOLO labels...')
    train_lbl_dir.mkdir(exist_ok=True)
    with open(coco_root / 'train' / 'annotations' / 'train.json') as f:
        coco = json.load(f)
    img_info = {img['id']: img for img in coco['images']}
    cat_to_idx = {cat['id']: i for i, cat in enumerate(coco['categories'])}
    ann_by_img = defaultdict(list)
    for ann in coco['annotations']:
        ann_by_img[ann['image_id']].append(ann)
    existing_stems = {p.stem for p in train_img_dir.iterdir() if p.suffix.lower() in img_exts}
    written = 0
    for img_id, anns in ann_by_img.items():
        info = img_info.get(img_id)
        if not info: continue
        stem = Path(info['file_name']).stem
        if stem not in existing_stems: continue
        W, H = info['width'], info['height']
        lines = []
        for ann in anns:
            cls_idx = cat_to_idx.get(ann['category_id'])
            if cls_idx is None: continue
            x, y, w, h = ann['bbox']
            cx=(x+w/2)/W; cy=(y+h/2)/H; nw=w/W; nh=h/H
            lines.append(f'{cls_idx} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        if lines:
            (train_lbl_dir / f'{stem}.txt').write_text('\n'.join(lines))
            written += 1
    print(f'✓ Wrote {written} YOLO label files')
else:
    print(f'✓ Labels already present: {len(list(train_lbl_dir.glob("*.txt")))} files')

# Build image lists with 85/15 train/val split
labelled_stems = {p.stem for p in train_lbl_dir.glob('*.txt')}
all_train = [p for p in train_img_dir.iterdir()
             if p.suffix.lower() in img_exts and p.stem in labelled_stems]
random.seed(42); random.shuffle(all_train)
n_val = max(1, int(len(all_train) * 0.15))
val_imgs   = all_train[:n_val]
train_imgs = all_train[n_val:]
test_imgs  = [p for p in test_img_dir.iterdir() if p.suffix.lower() in img_exts]

YOLO_DIR = LOCAL / 'yolo'
YOLO_DIR.mkdir(exist_ok=True)
(YOLO_DIR / 'train_list.txt').write_text('\n'.join(str(p) for p in train_imgs))
(YOLO_DIR / 'val_list.txt'  ).write_text('\n'.join(str(p) for p in val_imgs))
(YOLO_DIR / 'test_list.txt' ).write_text('\n'.join(str(p) for p in test_imgs))

total = len(train_imgs) + len(val_imgs) + len(test_imgs)
YAML_PATH = YOLO_DIR / 'dataset.yaml'
YAML_PATH.write_text(
    f'# Shezhen Tongue Dataset — YOLOv8\n'
    f'# {total} labelled images | 21 classes\n\n'
    f'train: {YOLO_DIR}/train_list.txt\n'
    f'val:   {YOLO_DIR}/val_list.txt\n'
    f'test:  {YOLO_DIR}/test_list.txt\n\n'
    f'nc: {len(CLASS_NAMES)}\n'
    f'names: {CLASS_NAMES}\n'
)

print(f'\n✓ dataset.yaml written')
print(f'  Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)} | Total: {total}')

In [ ]:
# ── Cell 7: TRAIN ─────────────────────────────────────────────────────────────
from ultralytics import YOLO
from pathlib import Path

YAML_PATH = Path('/content/shezhen_data/yolo/dataset.yaml')

print('=' * 60)
print('  Chemigran Tongue Diagnosis — YOLOv8 Training')
print('=' * 60)

model = YOLO('yolov8n.pt')   # nano — fast, good for this dataset size

results = model.train(
    data=str(YAML_PATH),
    epochs=80,
    batch=32,          # GPU can handle larger batches
    imgsz=640,
    device=0,          # GPU
    project='/content/runs',
    name='shezhen_train',
    exist_ok=True,

    # Tongue-optimised augmentation
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.3,
    fliplr=0.3,
    mosaic=0.8,
    mixup=0.1,

    # Optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=20,
    save=True,
    plots=True,
    verbose=True,
)

print('\n✅ Training complete!')

In [ ]:
# ── Cell 8: Validate on test set ──────────────────────────────────────────────
from ultralytics import YOLO
from pathlib import Path

best_pt = Path('/content/runs/shezhen_train/weights/tongue_yolo_best.pt')
best_model = YOLO(str(best_pt))

metrics = best_model.val(
    data='/content/shezhen_data/yolo/dataset.yaml',
    split='test',
    imgsz=640,
    device=0,
)

print(f'\n📊 Test Set Results:')
print(f'   mAP50     : {metrics.box.map50:.3f}')
print(f'   mAP50-95  : {metrics.box.map:.3f}')
print(f'   Precision : {metrics.box.mp:.3f}')
print(f'   Recall    : {metrics.box.mr:.3f}')

In [ ]:
# ── Cell 9: Save model back to Google Drive ───────────────────────────────────
import shutil
from pathlib import Path

best_pt = Path('/content/runs/shezhen_train/weights/tongue_yolo_best.pt')

# Save to Drive
drive_models = Path('/content/drive/MyDrive/tcm_platform_models')
drive_models.mkdir(exist_ok=True)
shutil.copy2(best_pt, drive_models / 'tongue_yolo_best.pt')
print(f'✓ Saved to Drive: {drive_models}/tongue_yolo_best.pt')

# Also available for direct download
from google.colab import files
print('\nDownloading tongue_yolo_best.pt to your computer...')
files.download(str(best_pt))

## ✅ Done!

**Next steps:**
1. Copy `tongue_yolo_best.pt` into `C:\Users\colli\OneDrive\Desktop\tcm_platform\models\`
2. Run `streamlit run app.py` in WSL with `conda activate in_silico`
3. The Visual AI Diagnosis tab will show **⚡ YOLOv8 + Claude Pipeline Active**

**Expected performance on this dataset:**
- mAP50 ~0.75–0.85 after 80 epochs on T4 GPU
- Inference: ~15ms per image (near real-time)